# Using `pyfwg` for direct morphing with `morph_epw_global`, `morph_epw_europe` and `morph_epw_cordex`

This notebook demonstrates the simplest and most direct way to use the `pyfwg` library: the `morph_epw_global`, `morph_epw_europe` and `morph_epw_cordex` functions.

These functions are designed as one-shot tools that provide full access to all of the Future Weather Generator's parameters without requiring the user to manage the multi-step `MorphingWorkflow` class. They handle parameter validation, temporary file management, and final file placement in a single call.

**Note:** `pyfwg` fully supports **FutureWeatherGenerator v4.0.x** (Global) and **v2.0.x** (Europe), which include a new keyword-based CLI, support for string-based parameter options (like `'AVG4P'` for interpolation), and additional output formats (like `'MET'` or `'CSV'`).

**New in pyfwg v0.4.0:** `pyfwg` also supports the **FWG Engine**, the shared engine of the latest distributions of the tool: **Global v5.x**, **Europe v3.x** and the **CORDEX regional** distributions. They are covered in the sections *Using `morph_epw_global` with the FWG Engine (Global v5.x)*, *Europe v3.x (FWG Engine)* and *Using `morph_epw_cordex`*.

## Configuration

The examples in this notebook run the Future Weather Generator, which is not distributed with `pyfwg`. Download its JAR files from the [official website](https://future-weather-generator.adai.pt/) and put them in a folder. Then, set the `FWG_JAR_DIR` environment variable to that folder before starting Jupyter, or replace `'path/to/jars'` with its path in the cell below. All the JAR paths of this notebook are built from `FWG_JAR_DIR`: if you use other versions of the tool, change the filenames where the paths are defined.

In the saved outputs, `<FWG_JAR_DIR>`, `<TEMP>` and `<HOME>` stand for the JAR folder, the temporary folder and the home folder of the computer where the notebook was run.

In [ ]:
import os

# Folder that contains the JAR files of the Future Weather Generator. Set the FWG_JAR_DIR
# environment variable, or replace 'path/to/jars' with the folder on your computer.
FWG_JAR_DIR = os.environ.get('FWG_JAR_DIR', 'path/to/jars')

## Understanding the Parameters

The `morph_epw_global`, `morph_epw_europe` and `morph_epw_cordex` functions are powerful because they expose all of the underlying tool's options. Here is a complete reference for all their arguments.

### Core Parameters

*   `epw_paths` (`Union[str, List[str]]`): **Required.** The path to a single EPW file (as a string) or a list of paths to multiple EPW files.
*   `fwg_jar_path` (`str`): **Required.** The absolute or relative path to the Future Weather Generator `.jar` file (e.g., `FutureWeatherGenerator_v4.0.2.jar` or `FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar`).

### Workflow Control Parameters

*   `output_dir` (`str`, default: `'./morphed_epws'`): The directory where the final morphed files will be saved.
*   `delete_temp_files` (`bool`, default: `True`): If `True`, the temporary folders are deleted after processing. Set to `False` for debugging.
*   `temp_base_dir` (`str`, default: `'./morphing_temp_results'`): The base directory where temporary folders will be created.
*   `fwg_show_tool_output` (`bool`, default: `False`): If `True`, prints the FWG tool's console output in real-time. Highly recommended for monitoring progress.
*   `fwg_params` (`Optional[Dict]`, default: `None`): A dictionary for providing a base set of FWG parameters. Any explicit `fwg_` argument will override the values in this dictionary.

### Future Weather Generator Tool Parameters

These arguments correspond directly to the options in the FWG command-line tool.

*   `fwg_gcms` (`Optional[List[str]]`, default: `None`): **Only valid for `morph_epw_global`**. A list of GCMs to use. If `None`, the full default list is used to create an ensemble.
*   `fwg_rcm_pairs` (`Optional[List[str]]`, default: `None`): **Only valid for `morph_epw_europe` and `morph_epw_cordex`**. List of GCM-RCM pairs to use. If `None`, the tool's default list is used.
*   `fwg_create_ensemble` (`bool`, default: `True`): If `True`, creates an ensemble from the selected GCMs.
*   `fwg_winter_sd_shift` (`float`, default: `0.0`): Temperature's standard deviation shift for the winter peak month. Range: -2.0 to 2.0.
*   `fwg_summer_sd_shift` (`float`, default: `0.0`): Temperature's standard deviation shift for the summer peak month. Range: -2.0 to 2.0.
*   `fwg_month_transition_hours` (`int`, default: `72`): Number of hours to smooth the transition between months. Range: 0 to 336.
*   `fwg_use_multithreading` (`bool`, default: `True`): If `True`, performs computations in multithread.
*   `fwg_interpolation_method_id` (`Union[int, str]`, default: `0`): Selects the grid method. **Legacy options:** `0` (IDW), `1` (AVG4P), `2` (NP). **V4/Europe v2 options:** you can use strings: `'IDW'`, `'BI'`, `'AVG4P'`, `'NP'`. **FWG Engine options:** also `'SCI'`, `'SGBI'` and `'STBI'` (or `4`, `5` and `6`).
*   `fwg_limit_variables` (`bool`, default: `True`): If `True`, bounds each generated variable to its physical limits.
*   `fwg_solar_hour_adjustment` (`Union[int, str]`, default: `1`): Option for solar hour adjustment. **Legacy options:** `0`, `1`, `2`. **V4 options:** `'None'`, `'By_Month'`, `'By_Day'`. Ignored by the FWG Engine.
*   `fwg_diffuse_irradiation_model` (`Union[int, str]`, default: `1`): Option for diffuse solar irradiation model. **Legacy options:** `0`, `1`, `2`. **V4 options:** `'Ridley_Boland_Lauret_2010'`, `'Engerer_2015'`, `'Paulescu_Blaga_2019'`. With the FWG Engine, use `fwg_decomposition_method` instead.
*   `fwg_add_uhi` (`Optional[bool]`, default: `None`): Option to pre-process the Urban Heat Island (UHI) effect. If `None`, it is applied with the versions that support it (Global v3/v4 and Europe v1/v2). The FWG Engine does not support it, so `morph_epw_cordex` has no UHI or LCZ arguments.
*   `fwg_epw_original_lcz` (`int`, default: `14`): The Local Climate Zone (LCZ) of the original EPW. Range: 1 to 17.
*   `fwg_target_uhi_lcz` (`int`, default: `1`): The target LCZ of the generated EPW. Range: 1 to 17.
*   `fwg_output_type` (`str`, default: `'EPW'`): **New in V4/Europe v2.** Format of output files: `'EPW'`, `'SPAIN_MET'`, or `'PORTUGAL_CSV'`.
*   `fwg_version` (`Optional[str]`, default: `None`): Force a specific CLI version (e.g., `'4'`). If `None`, `pyfwg` auto-detects it from the JAR (from its filename or, for FWG Engine JARs, from the packaged distribution manifest).

### FWG Engine Parameters (Global v5.x, Europe v3.x and CORDEX regional)

These arguments are only available for the distributions that run on the FWG Engine. They default to `None`, which keeps the default of the tool.

*   `fwg_scenarios` (`Optional[List[str]]`, default: `None`): Subset of scenarios to generate (e.g., `['ssp245', 'ssp585']` or `['rcp85']`). If `None`, all the scenarios of the distribution are generated.
*   `fwg_timeframes` (`Optional[List[Union[int, str]]]`, default: `None`): Subset of future periods to generate (e.g., `[2050]`). If `None`, all the periods of the distribution are generated.
*   `fwg_decomposition_method` (`Optional[str]`, default: `None`): Solar decomposition model used to split global radiation into direct and diffuse components (tool default: `'MULTIMODEL_CONSTRAINED_ENSEMBLE'`).
*   `fwg_illuminance_method` (`Optional[str]`, default: `None`): `'PEREZ_1990'` (tool default) or `'SOURCE_CALIBRATED_LUMINOUS_EFFICACY'`.
*   `fwg_infrared_method` (`Optional[str]`, default: `None`): `'FWG_CURRENT'` (tool default) or `'CLARK_ALLEN_WALTON'`. Available from FWG Engine v1.1 (e.g., Global v5.2).
*   `fwg_morphing_method` (`Optional[str]`, default: `None`): `'CLASSIC_MONTHLY'` (tool default) or `'DISTRIBUTION_AWARE'` (experimental).
*   `fwg_uncertainty_variant` (`Optional[str]`, default: `None`): Uncertainty case of the climate-change signal: `'CENTRAL'` (tool default), `'PLUS_ONE_SIGMA'`, `'MINUS_ONE_SIGMA'`, `'WARM_DRY'`, `'WARM_HUMID'`, `'HIGH_SOLAR'` or `'LOW_SOLAR'`. Non-central cases cannot be combined with non-zero `fwg_winter_sd_shift` or `fwg_summer_sd_shift`.
*   `fwg_uncertainty_sigma` (`Optional[float]`, default: `None`): Standard-deviation multiplier of the uncertainty case (tool default: `1.0`).
*   `fwg_output_profile` (`Optional[str]`, default: `None`): Reports written alongside the weather files: `'MINIMAL'` (weather files only), `'STANDARD'` (tool default), `'FULL'` or `'CUSTOM'`. They remain in the temporary directory.
*   `fwg_model_unzipped_folder` (`Optional[str]`, default: `None`): Folder where the tool caches the unzipped climate-model data, so that later runs can reuse it.
*   `fwg_extra_args` (`Optional[Dict[str, Any]]`, default: `None`): Any other option of the tool, passed as `-key=value` (e.g., `{'threads': 4}`). Also available for Global v4.x and Europe v2.x.

## Using `morph_epw_global`

### Step 1: Imports and Setup

First, we'll import the necessary function and set up our file paths. 

**Important:** `jar_path` is built from `FWG_JAR_DIR`, set in the *Configuration* section at the top of the notebook. If your Future Weather Generator `.jar` file has a different name, change it in the cell below. `pyfwg` will automatically detect the version (v3 or v4) and adapt the CLI command accordingly.

In [1]:
import os
from pyfwg import get_fwg_parameters_info, morph_epw_global, get_available_lczs, check_lcz_availability, DEFAULT_GLOBAL_GCMS

# --- Configuration ---
# Define the path to the EPW file to be processed. You can also define a list of paths to process multiple EPW files.
epw_file = 'epws/w_pattern/sevilla_uhi-type-1.epw'

# V4 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator_v4.0.2.jar')

### Exploring FWG Parameters

Before configuring the workflow, you might want to know which parameters are available and what values they accept. This is especially useful for the new string-based options in V4.

You can use the `get_fwg_parameters_info` utility to get a detailed list of all parameters.

In [2]:
import json
# Get the info and print it nicely
info = get_fwg_parameters_info()
print(json.dumps(info, indent=4))

{
    "fwg_gcms": {
        "description": "List of Global Climate Models (GCMs) to use for the Global tool.",
        "allowed_values": [
            "BCC_CSM2_MR",
            "CAS_ESM2_0",
            "CMCC_ESM2",
            "CNRM_CM6_1",
            "CNRM_CM6_1_HR",
            "CNRM_ESM2_1",
            "CanESM5",
            "CanESM5_1",
            "CanESM5_CanOE",
            "EC_Earth3",
            "EC_Earth3_Veg",
            "EC_Earth3_Veg_LR",
            "FGOALS_g3",
            "GFDL_ESM4",
            "GISS_E2_1_G",
            "GISS_E2_1_H",
            "GISS_E2_2_G",
            "IPSL_CM6A_LR",
            "MIROC6",
            "MIROC_ES2H",
            "MIROC_ES2L",
            "MRI_ESM2_0",
            "UKESM1_0_LL"
        ],
        "default": "All available GCMs",
        "applies_to": "Global"
    },
    "fwg_rcm_pairs": {
        "description": "List of GCM-RCM model pairs to use for the Europe-specific and CORDEX regional tools.",
        "allowed_values": [


### Step 2: Pre-flight Checks (Recommended). Check Available LCZs.

Before running a long morphing process, it's a good practice to validate your parameters. `pyfwg` provides utility functions to help with this.

The `fwg_epw_original_lcz` and `fwg_target_uhi_lcz` parameters depend on which Local Climate Zones are available for a specific weather file location. We can find this out using `get_available_lczs`.

In [3]:
available_zones = get_available_lczs(
    epw_paths=epw_file,
    fwg_jar_path=jar_path,
    # In java_class_path_prefix, use 'futureweathergenerator' for the global scope FWG tool, otherwise 'futureweathergenerator_europe'
    java_class_path_prefix='futureweathergenerator',
    show_tool_output=False,
)

2025-12-30 20:28:04 - INFO - --- Fetching available LCZs for 1 EPW file(s) ---
2025-12-30 20:28:04 - INFO - Checking LCZ pair (Original: 0, Target: 0) availability for sevilla_uhi-type-1.epw...
2025-12-30 20:28:05 - INFO - --- Applying UHI effect to sevilla_uhi-type-1.epw ---
2025-12-30 20:28:05 - INFO - Executing command: java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -u -epw=<TEMP>\tmpm08u8xr7\sevilla_uhi-type-1.epw -output_folder=<TEMP>\tmpm08u8xr7\ -uhi=true:0:0 -output_type=EPW
2025-12-30 20:28:23 - INFO - Available LCZs for 'sevilla_uhi-type-1.epw': [2, 3, 6, 8, 9, 11, 12, 13, 14, 16]


In [4]:
# Print the results in a user-friendly format
for filename, lcz_list in available_zones.items():
    print(f"\nAvailable LCZs for: {filename}")
    if lcz_list:
        print(f"  {lcz_list}")
    else:
        print("  Could not be determined.")


Available LCZs for: sevilla_uhi-type-1.epw
  [2, 3, 6, 8, 9, 11, 12, 13, 14, 16]


### Step 3: Run the Morphing Process

Now, we can call the `morph_epw_global` function. We will override a few of the default parameters to demonstrate how to customize the process. To speed up the process, let's use just one GCM. If you don't know which are these, you can import the list containing the default models:

In [5]:
from pyfwg import get_fwg_parameters_info, DEFAULT_GLOBAL_GCMS
print(DEFAULT_GLOBAL_GCMS)

{'GISS_E2_1_H', 'CNRM_CM6_1', 'MIROC6', 'IPSL_CM6A_LR', 'CMCC_ESM2', 'EC_Earth3_Veg', 'EC_Earth3_Veg_LR', 'BCC_CSM2_MR', 'CanESM5_CanOE', 'FGOALS_g3', 'GISS_E2_2_G', 'CanESM5_1', 'GFDL_ESM4', 'CNRM_CM6_1_HR', 'MRI_ESM2_0', 'MIROC_ES2H', 'EC_Earth3', 'MIROC_ES2L', 'CanESM5', 'UKESM1_0_LL', 'GISS_E2_1_G', 'CAS_ESM2_0', 'CNRM_ESM2_1'}


So let's use for instance "CanESM5". Note that in **V4**, we can use descriptive strings for parameters like `fwg_interpolation_method_id`.

In [6]:
# The validation of parameters is executed automatically inside the function.
created_files_custom = morph_epw_global(
    epw_paths=epw_file,
    fwg_jar_path=jar_path,
    output_dir='./custom_output_global',
    fwg_show_tool_output=True,      # See the tool's progress in real-time
    fwg_gcms=['CanESM5'],           # Use only one specific GCM for speed
    fwg_interpolation_method_id='AVG4P',  # V4 feature: Use string names!
    fwg_epw_original_lcz=2,         # Use a validated LCZ from the check above
    fwg_target_uhi_lcz=3,          # Use a validated LCZ from the check above
    fwg_version='4'                 # Explicitly force V4 syntax
)

2025-12-30 20:28:24 - INFO - --- Starting Direct Global Morphing Process ---
2025-12-30 20:28:24 - INFO - --- Step 2: Configuring and Previewing Morphing Plan ---
2025-12-30 20:28:24 - INFO - Validating LCZ availability for sevilla_uhi-type-1.epw...
2025-12-30 20:28:24 - INFO - Checking LCZ pair (Original: 2, Target: 3) availability for sevilla_uhi-type-1.epw...
2025-12-30 20:28:24 - INFO - --- Applying UHI effect to sevilla_uhi-type-1.epw ---



          MORPHING CONFIGURATION & PREVIEW
  - FWG JAR Path: <FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar
  - Final Output Directory: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global
  - EPWs to be Morphed (1 files):
    - sevilla_uhi-type-1.epw

  For input file: sevilla_uhi-type-1.epw
    -> Generated 'ssp126_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ssp126_2050.epw
    -> Generated 'ssp245_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ssp245_2050.epw
    -> Generated 'ssp370_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ssp370_2050.epw
    -> Generated 'ssp585_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ssp585_2050.epw
    -> Generated 'ssp126_2080.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ssp126_2080.epw
    -> Generated 'ssp245_2080.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_ou

2025-12-30 20:28:24 - INFO - Executing command: java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -u -epw=<TEMP>\tmp6c1xil_q\sevilla_uhi-type-1.epw -output_folder=<TEMP>\tmp6c1xil_q\ -uhi=true:2:3 -output_type=EPW
2025-12-30 20:28:40 - INFO - UHI effect applied successfully.
2025-12-30 20:28:40 - INFO - LCZ pair (Original: 2, Target: 3) is available.
2025-12-30 20:28:40 - INFO - Copied input file to temporary directory: ./morphing_temp_results\sevilla_uhi-type-1\sevilla_uhi-type-1.epw



-------------------- Executing FWG for sevilla_uhi-type-1.epw --------------------
  Full Command (for reference): java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_v4.0.2.jar" -epw=D:\Python\pyfwg\pyfwg\tutorials\epws\w_pattern\sevilla_uhi-type-1.epw -output_folder=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results\sevilla_uhi-type-1\ -models=CanESM5 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=AVG4P -solar_correction=By_Month -diffuse_method=Engerer_2015 -uhi=true:2:3 -output_type=EPW
  --- FWG Real-time Output ---


2025-12-30 20:29:00 - INFO - Direct global morphing complete. 18 files created in D:\Python\pyfwg\pyfwg\tutorials\custom_output_global


  --- End of FWG Output ---


In [7]:
print("\nSuccessfully created files:")
for f in created_files_custom:
    print(f)


Successfully created files:
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2050.epw
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2050.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2080.epw
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp126_2080.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp245_2050.epw
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp245_2050.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_ssp245_2080.epw
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Origina

### Expected Output

When you run the code above with **V4**, you should see:
1. A detailed printout of the **keyword-based command** being executed (e.g., `-epw=... -models=CanESM5`).
2. The real-time console output from the Future Weather Generator tool.
3. A final message listing all the created files in the `./custom_output_global` directory.

Let's delete the files we just generated.

In [8]:
import shutil
if os.path.exists('custom_output_global'):
    shutil.rmtree('custom_output_global')

## Using `morph_epw_global` with the FWG Engine (Global v5.x)

Since version 5, the Global tool runs on the **FWG Engine**, the engine shared by the latest distributions of the tool (Global v5.x, Europe v3.x and the CORDEX regional distributions). You can keep using the same `morph_epw_global` function: `pyfwg` detects the version from the JAR and builds the new command line for you.

The FWG Engine brings some changes and new options:

*   **Scenarios and future periods**: With `fwg_scenarios` and `fwg_timeframes`, you can generate only the scenarios and periods you need (e.g., `['ssp585']` and `[2050]`), which saves a lot of time.
*   **Uncertainty cases**: `fwg_uncertainty_variant` generates boundary cases of the climate-change signal (e.g., `'WARM_DRY'` or `'PLUS_ONE_SIGMA'`) besides the central signal (`'CENTRAL'`, the default).
*   **Climate-data cache**: In each run, the tool unzips the climate-model data packaged in the JAR. With `fwg_model_unzipped_folder`, the unzipped data is cached in a folder and reused in later runs. Since it can take several hundred MB, keep this folder outside your project.
*   **No Urban Heat Island (UHI) effect**: The FWG Engine does not support the UHI effect, so the LCZ arguments are not used. `fwg_add_uhi` defaults to `None`, which applies the UHI effect only with the versions that support it (Global v3/v4 and Europe v1/v2). An explicit `fwg_add_uhi=True` is ignored with a warning.
*   **Other options**: Any other option of the tool can be passed with `fwg_extra_args` (see below).

### Step 1: Imports and Setup

**Important:** `jar_path_v5` is built from `FWG_JAR_DIR`, set in the *Configuration* section at the top of the notebook. If your Global v5 `.jar` file has a different name, change it in the cell below. We will morph the same EPW file as before.

In [3]:
import os
import shutil
from pyfwg import morph_epw_global, get_fwg_distribution_info, detect_fwg_version

# Global v5 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path_v5 = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar')

# Folder to cache the unzipped climate-model data across runs. It can take several hundred MB,
# so we keep it outside the project folder (here, in the user's home folder).
model_cache_dir = os.path.join(os.path.expanduser('~'), 'fwg_model_cache', 'cmip6_global')

### Step 2: Inspect the Distribution

FWG Engine JARs package a manifest with the scenarios, future periods and climate models they provide. You can read it with `get_fwg_distribution_info`, which does not need to run Java. `pyfwg` also uses this information to validate your configuration before running the tool.

In [4]:
info = get_fwg_distribution_info(jar_path_v5)

print(f"Detected version: {detect_fwg_version(jar_path_v5)}")
print(f"Distribution: {info['name']} v{info['version']} (FWG Engine v{info['engine_version']})")
print(f"Scenarios: {info['scenarios']}")
print(f"Future periods: {info['timeframes']}")
print(f"Climate models ({len(info['models'])}): {info['models']}")

Detected version: 5
Distribution: FWG CMIP6 Global v5.2.0 (FWG Engine v1.1.0)
Scenarios: ['ssp126', 'ssp245', 'ssp370', 'ssp585']
Future periods: ['2050', '2080']
Climate models (23): ['BCC-CSM2-MR', 'CanESM5', 'CanESM5-1', 'CanESM5-CanOE', 'CAS-ESM2-0', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-g3', 'GFDL-ESM4', 'GISS-E2-1-G', 'GISS-E2-1-H', 'GISS-E2-2-G', 'IPSL-CM6A-LR', 'MIROC-ES2H', 'MIROC-ES2L', 'MIROC6', 'MRI-ESM2-0', 'UKESM1-0-LL']


`get_fwg_distribution_info` returns `None` for JARs prior to the FWG Engine (Global v3/v4 and Europe v1/v2). Besides the keys printed above, the dictionary includes `ensemble_models` (the models used by default to create the ensemble) and `model_details` (the scenarios and periods available for each model).

Note that the FWG Engine writes the model names with hyphens (e.g., `'BCC-CSM2-MR'`). `pyfwg` accepts both hyphens and underscores, so the names in `DEFAULT_GLOBAL_GCMS` (e.g., `'BCC_CSM2_MR'`) also work.

### Step 3: Run the Morphing Process

Now, we call `morph_epw_global` with the new options. To keep the example fast, we use two GCMs (the FWG Engine needs at least two models to create an ensemble), a single scenario and a single future period. We also generate the warm and dry boundary case of the climate-change signal (`'WARM_DRY'`) instead of the central one.

In [5]:
created_files_v5 = morph_epw_global(
    epw_paths=epw_file,
    fwg_jar_path=jar_path_v5,
    output_dir='./custom_output_global_v5',
    fwg_show_tool_output=False,                # The FWG Engine is very verbose: set it to True to see its output in real time
    fwg_gcms=['BCC_CSM2_MR', 'CanESM5'],       # Two GCMs, so that the ensemble can be created
    fwg_scenarios=['ssp585'],                  # Only generate this scenario...
    fwg_timeframes=[2050],                     # ...and this future period
    fwg_uncertainty_variant='WARM_DRY',        # Uncertainty case (default: 'CENTRAL')
    fwg_model_unzipped_folder=model_cache_dir  # Reuse the unzipped climate data across runs
)

2026-10-17 22:11:04 - INFO - --- Starting Direct Global Morphing Process ---
2026-10-17 22:11:04 - INFO - --- Step 2: Configuring and Previewing Morphing Plan ---
2026-10-17 22:11:04 - INFO - Auto-detected FWG version: 5
2026-10-17 22:11:04 - INFO - Detected FWG CMIP6 Global v5.2.0 (FWG Engine 1.1.0).
2026-10-17 22:11:04 - INFO - Copied input file to temporary directory: ./morphing_temp_results\sevilla_uhi-type-1\sevilla_uhi-type-1.epw



          MORPHING CONFIGURATION & PREVIEW
  - FWG JAR Path: <FWG_JAR_DIR>\FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar
  - Final Output Directory: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global_v5
  - EPWs to be Morphed (1 files):
    - sevilla_uhi-type-1.epw

  For input file: sevilla_uhi-type-1.epw
    -> Generated 'ssp585_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_global_v5\ssp585_2050.epw
Configuration set. Call execute_morphing() to start the process.

-------------------- Executing FWG for sevilla_uhi-type-1.epw --------------------
  Full Command (for reference): java -jar <FWG_JAR_DIR>\FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar -g -epw=D:\Python\pyfwg\pyfwg\tutorials\epws\w_pattern\sevilla_uhi-type-1.epw -output_folder=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results\sevilla_uhi-type-1\ -output_layout=ORGANIZED -models=BCC_CSM2_MR,CanESM5 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=

2026-10-17 22:11:38 - INFO - Direct global morphing complete. 2 files created in D:\Python\pyfwg\pyfwg\tutorials\custom_output_global_v5


In [6]:
print("\nSuccessfully created files:")
for f in created_files_v5:
    print(f)


Successfully created files:
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global_v5\ESP_-_SEVILLA_Ensemble_ssp585_2050_warm-dry.epw
D:\Python\pyfwg\pyfwg\tutorials\custom_output_global_v5\ESP_-_SEVILLA_Ensemble_ssp585_2050_warm-dry.stat


Only the weather and `.stat` files of the selected scenario and period are moved to the output folder, with the filenames given by the tool. Note that, for non-central uncertainty cases, the FWG Engine adds the case to the filenames (e.g., `_warm-dry`). The other files written by the FWG Engine (e.g., the corrected baseline weather file, reports and manifests) remain in the temporary directory, which is deleted by default (`delete_temp_files=True`).

#### Passing Other Options with `fwg_extra_args`

The tool has more options than the `fwg_` arguments of `pyfwg`. You can pass any of them with `fwg_extra_args`, a dictionary of option names and values that are added to the command line as `-key=value`. For example, to limit the number of CPU threads used by the tool:

```python
created_files_v5 = morph_epw_global(
    epw_paths=epw_file,
    fwg_jar_path=jar_path_v5,
    fwg_gcms=['BCC_CSM2_MR', 'CanESM5'],
    fwg_extra_args={'threads': 4}  # Added to the command line as -threads=4
)
```

The options that `pyfwg` already sets (e.g., `epw`, `output_folder` or `models`) cannot be passed through `fwg_extra_args`; use the corresponding `fwg_` argument instead. `fwg_extra_args` is also available for Global v4.x and Europe v2.x.

Let's delete the files we just generated. The climate-data cache is kept so that later runs can reuse it: delete `model_cache_dir` when you no longer need it.

In [7]:
if os.path.exists('custom_output_global_v5'):
    shutil.rmtree('custom_output_global_v5')

## Using `morph_epw_europe`

>**FYI**: the usage of `morph_epw_europe` is very similar to the global version. The only differences are:

>* Instead of argument `fwg_gcms`, you need to use `fwg_rcm_pairs`.
>* Instead of variable `DEFAULT_GLOBAL_GCMS`, the available models are in variable `DEFAULT_EUROPE_RCMS`.
>* Future scenarios are RCP instead of SSP.
>* You have to remember using the `jar_path` to the FutureWeatherGenerator_Europe_vx.x.x.jar file.

### Step 1: Imports and Setup

**Important:** `jar_path_europe` is built from `FWG_JAR_DIR`, set in the *Configuration* section at the top of the notebook. If your European version JAR (v1.x for legacy or v2.x for the update) has a different name, change it in the cell below.

In [9]:
from pyfwg import get_fwg_parameters_info, morph_epw_europe

# Europe v2 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path_europe = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator_Europe_v2.0.2.jar')

### Step 2: Run the Morphing Process

Now, we can call the `morph_epw_europe` function. In **Europe v2**, we can also specify a different output format like `'MET'`.

In [10]:
from pyfwg import get_fwg_parameters_info, DEFAULT_EUROPE_RCMS

created_files_custom = morph_epw_europe(
    epw_paths=epw_file,
    fwg_jar_path=jar_path_europe,
    output_dir='./custom_output_europe',
    fwg_show_tool_output=True,
    fwg_rcm_pairs=[list(DEFAULT_EUROPE_RCMS)[0]],  # Use the first pair for speed
    fwg_interpolation_method_id='NP',              # Nearest Point
    fwg_epw_original_lcz=2,
    fwg_target_uhi_lcz=3,
    fwg_output_type='SPAIN_MET'                         # Output formatted as MET files
)

2025-12-30 20:29:00 - INFO - --- Starting Europe-Specific Direct Morphing Process ---
2025-12-30 20:29:00 - INFO - --- Step 2: Configuring and Previewing Morphing Plan ---
2025-12-30 20:29:00 - INFO - Auto-detected FWG version: 2
2025-12-30 20:29:00 - INFO - Validating LCZ availability for sevilla_uhi-type-1.epw...
2025-12-30 20:29:00 - INFO - Checking LCZ pair (Original: 2, Target: 3) availability for sevilla_uhi-type-1.epw...
2025-12-30 20:29:00 - INFO - --- Applying UHI effect to sevilla_uhi-type-1.epw ---



          MORPHING CONFIGURATION & PREVIEW
  - FWG JAR Path: <FWG_JAR_DIR>\FutureWeatherGenerator_Europe_v2.0.2.jar
  - Final Output Directory: D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe
  - EPWs to be Morphed (1 files):
    - sevilla_uhi-type-1.epw

  For input file: sevilla_uhi-type-1.epw
    -> Generated 'rcp26_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\rcp26_2050.epw
    -> Generated 'rcp45_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\rcp45_2050.epw
    -> Generated 'rcp85_2050.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\rcp85_2050.epw
    -> Generated 'rcp26_2080.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\rcp26_2080.epw
    -> Generated 'rcp45_2080.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\rcp45_2080.epw
    -> Generated 'rcp85_2080.epw' will be moved to: D:\Python\pyfwg\pyfwg\tutorials\custom_output

2025-12-30 20:29:00 - INFO - Executing command: java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_Europe_v2.0.2.jar" -u -epw=<TEMP>\tmp4znojmmf\sevilla_uhi-type-1.epw -output_folder=<TEMP>\tmp4znojmmf\ -uhi=true:2:3 -output_type=EPW
2025-12-30 20:29:15 - INFO - UHI effect applied successfully.
2025-12-30 20:29:15 - INFO - LCZ pair (Original: 2, Target: 3) is available.
2025-12-30 20:29:15 - INFO - Copied input file to temporary directory: ./morphing_temp_results_europe\sevilla_uhi-type-1\sevilla_uhi-type-1.epw



-------------------- Executing FWG for sevilla_uhi-type-1.epw --------------------
  Full Command (for reference): java -jar "<FWG_JAR_DIR>\FutureWeatherGenerator_Europe_v2.0.2.jar" -epw=D:\Python\pyfwg\pyfwg\tutorials\epws\w_pattern\sevilla_uhi-type-1.epw -output_folder=D:\Python\pyfwg\pyfwg\tutorials\morphing_temp_results_europe\sevilla_uhi-type-1\ -models=MOHC_HadGEM2_ES_DMI_HIRHAM5 -ensemble=true -temp_shift_winter=0.0 -temp_shift_summer=0.0 -smooth_hours=72 -multithread=true -grid_interpolation_method=NP -solar_correction=By_Month -diffuse_method=Engerer_2015 -uhi=true:2:3 -output_type=SPAIN_MET
  --- FWG Real-time Output ---


2025-12-30 20:29:53 - WARNING - PermissionError deleting ./morphing_temp_results_europe\sevilla_uhi-type-1. Retrying in 0.5s... (Attempt 1/5)


  --- End of FWG Output ---


2025-12-30 20:29:53 - WARNING - PermissionError deleting ./morphing_temp_results_europe\sevilla_uhi-type-1. Retrying in 0.5s... (Attempt 2/5)
2025-12-30 20:29:54 - INFO - Europe morphing complete. 7 files created in D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe


In [11]:
print("\nSuccessfully created files:")
for f in created_files_custom:
    print(f)


Successfully created files:
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp26_2050.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp26_2080.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp45_2050.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp45_2080.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp85_2050.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\ESP_-_SEVILLA_Ensemble_UHI_Target_LCZ-3_Original_LCZ-2_rcp85_2080.stat
D:\Python\pyfwg\pyfwg\tutorials\custom_output_europe\sevilla_uhi-type-1.epw


Let's delete the files we just generated.

In [12]:
if os.path.exists('custom_output_europe'):
    shutil.rmtree('custom_output_europe')

### Europe v3.x (FWG Engine)

Since version 3, the Europe-specific tool (`FutureWeatherGenerator-CORDEX-CMIP5-Europe-EUR-11-v3.X.X.jar`) also runs on the FWG Engine. Use the same `morph_epw_europe` function with the FWG Engine arguments shown in the Global v5.x section above. Remember that the scenarios are RCPs (e.g., `['rcp45', 'rcp85']`) and the models are GCM-RCM pairs.

> **Note:** The Europe v3 JAR was not available when this notebook was run, so the following cells have not been executed and their output is not shown.

In [ ]:
# Europe v3 JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename if you use another version.
jar_path_europe_v3 = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator-CORDEX-CMIP5-Europe-EUR-11-v3.2.0.jar')

# Use a separate climate-data cache for each distribution of the tool.
model_cache_dir_europe = os.path.join(os.path.expanduser('~'), 'fwg_model_cache', 'cordex_eur11')

created_files_europe_v3 = morph_epw_europe(
    epw_paths=epw_file,
    fwg_jar_path=jar_path_europe_v3,
    output_dir='./custom_output_europe_v3',
    fwg_rcm_pairs=['ICHEC_EC_EARTH_SMHI_RCA4', 'MPI_M_MPI_ESM_LR_SMHI_RCA4'],  # Two pairs to create the ensemble
    fwg_scenarios=['rcp45', 'rcp85'],
    fwg_timeframes=[2050],
    fwg_model_unzipped_folder=model_cache_dir_europe
)

print("\nSuccessfully created files:")
for f in created_files_europe_v3:
    print(f)

Let's delete the files we just generated.

In [ ]:
if os.path.exists('custom_output_europe_v3'):
    shutil.rmtree('custom_output_europe_v3')

## Using `morph_epw_cordex`

The **CORDEX regional distributions** of the Future Weather Generator also run on the FWG Engine. Each one covers a CORDEX domain: Middle East and North Africa (`MNA-22`), Africa (`AFR-22`), North America (`NAM-44`), Central America (`CAM-44`), South America (`SAM-44`), West & South Asia (`WAS-22`), Southeast Asia (`SEA-22`) and Australasia (`AUS-22`). Use them with the `morph_epw_cordex` function, which works like `morph_epw_europe` (`fwg_rcm_pairs` and RCP scenarios), with two differences:

*   **Region detection**: The region is detected from the JAR, and only its scenarios and GCM-RCM pairs are accepted. They are listed in `CORDEX_REGIONS`.
*   **No UHI**: These distributions do not support the Urban Heat Island effect, so `morph_epw_cordex` has no UHI or LCZ arguments.

Let's see what each region provides:

In [8]:
from pyfwg import morph_epw_cordex, CORDEX_REGIONS

for domain, region in CORDEX_REGIONS.items():
    print(f"{domain} ({region['name']}): scenarios {region['scenarios']}, {len(region['rcm_pairs'])} GCM-RCM pairs")

print("\nGCM-RCM pairs of MNA-22:", sorted(CORDEX_REGIONS['MNA-22']['rcm_pairs']))

EUR-11 (Europe): scenarios ['rcp26', 'rcp45', 'rcp85'], 7 GCM-RCM pairs
MNA-22 (Middle East and North Africa): scenarios ['rcp85'], 2 GCM-RCM pairs
AFR-22 (Africa): scenarios ['rcp26', 'rcp85'], 3 GCM-RCM pairs
NAM-44 (North America): scenarios ['rcp45', 'rcp85'], 2 GCM-RCM pairs
CAM-44 (Central America): scenarios ['rcp26', 'rcp45', 'rcp85'], 3 GCM-RCM pairs
SAM-44 (South America): scenarios ['rcp26', 'rcp45', 'rcp85'], 5 GCM-RCM pairs
WAS-22 (West & South Asia): scenarios ['rcp26', 'rcp85'], 2 GCM-RCM pairs
SEA-22 (Southeast Asia): scenarios ['rcp45', 'rcp85'], 2 GCM-RCM pairs
AUS-22 (Australasia): scenarios ['rcp26', 'rcp85'], 3 GCM-RCM pairs

GCM-RCM pairs of MNA-22: ['ICHEC_EC_EARTH_SMHI_RCA4', 'NOAA_GFDL_GFDL_ESM2M_SMHI_RCA4']


### Run the Morphing Process

The weather file must be located within the domain of the distribution. Our example file (Seville, Spain) lies within the CORDEX Middle East and North Africa domain (`MNA-22`), which only provides the `rcp85` scenario. For a weather file located elsewhere, use the JAR of its region (e.g., `NAM-44` for a weather file of Toronto).

We also use a separate climate-data cache (`fwg_model_unzipped_folder`) for each distribution, since some GCM-RCM pairs (e.g., `ICHEC_EC_EARTH_SMHI_RCA4`) are provided by several regions.

> **Note:** The CORDEX regional JARs were not available when this notebook was run, so the following cells have not been executed and their output is not shown.

In [ ]:
# CORDEX regional JAR in FWG_JAR_DIR (set at the top of the notebook). Change the filename for another region or version.
jar_path_mna = os.path.join(FWG_JAR_DIR, 'FutureWeatherGenerator-CORDEX-CMIP5-Middle-East-and-North-Africa-MNA-22-v1.2.0.jar')

model_cache_dir_mna = os.path.join(os.path.expanduser('~'), 'fwg_model_cache', 'cordex_mna22')

created_files_cordex = morph_epw_cordex(
    epw_paths=epw_file,
    fwg_jar_path=jar_path_mna,
    output_dir='./custom_output_cordex',
    fwg_rcm_pairs=sorted(CORDEX_REGIONS['MNA-22']['rcm_pairs']),  # Both GCM-RCM pairs of the region
    fwg_timeframes=[2050],  # MNA-22 only provides 'rcp85', so there is no need to select the scenarios
    fwg_model_unzipped_folder=model_cache_dir_mna
)

print("\nSuccessfully created files:")
for f in created_files_cordex:
    print(f)

Let's delete the files we just generated.

In [ ]:
if os.path.exists('custom_output_cordex'):
    shutil.rmtree('custom_output_cordex')